# Vue2 + Vue3 框架 · 第 05 天

## 自定义指令、插槽与 Vue Router 入门

> 本篇依据第 05 天的 MD 笔记、PPT 和随堂代码整理。当天内容把组件能力继续向前推进：用自定义指令封装 DOM 操作，用插槽让组件结构可定制，再用 Vue Router 在单页应用中完成页面切换。

本天的主线是“封装”：

- 把重复的 DOM 操作封装为自定义指令。
- 把不确定的组件结构交给插槽。
- 把表格、标签、弹框封装成可复用组件。
- 把路径和页面组件映射起来，建立 SPA 的路由结构。

## 一、学习目标与知识地图

| 模块 | 关键知识 | 最适合的场景 |
| --- | --- | --- |
| 自定义指令 | 局部/全局注册、指令钩子、`binding.value` | 聚焦、加载蒙层等 DOM 操作 |
| 默认插槽 | `<slot>`、后备内容 | 组件中只有一块内容需要自定义 |
| 具名插槽 | `name`、`v-slot`、`#` | 组件中有多块结构需要自定义 |
| 作用域插槽 | `slot` 绑定属性、`#default="obj"` | 表格行由外部决定显示内容 |
| 商品列表 | `MyTag`、`MyTable` | 练习指令、`v-model` 和作用域插槽 |
| SPA 与路由 | VueRouter、路由规则、导航、出口 | 按路径切换页面组件 |

### 选择工具的原则

- 需要直接操作元素：考虑自定义指令或 `$refs`。
- 需要外部替换组件内部结构：使用插槽。
- 需要外部决定每行或每项的展示：使用作用域插槽。
- 需要地址栏路径和页面组件建立关系：使用 Vue Router。

## 二、自定义指令

### 2.1 什么是自定义指令

`v-html`、`v-if`、`v-bind`、`v-on` 都是 Vue 内置指令。开发者也可以注册自己的指令，把一组重复的 DOM 操作封装起来。

自定义指令适合做“元素级”的增强，例如：

- 页面打开后自动聚焦。
- 数据请求时给容器添加加载蒙层。
- 根据指令传入的颜色设置文字颜色。

指令名称在注册时不写 `v-` 前缀，使用时才写 `v-`：

```js
// 注册名：focus
Vue.directive('focus', { /* ... */ })

// 使用名：v-focus
<input v-focus>
```

### 2.2 全局注册与局部注册

全局注册放在 `main.js`，项目中任何组件都可以使用：

```js
// main.js
import Vue from 'vue'

Vue.directive('focus', {
  inserted (el) {
    el.focus()
  }
})
```

局部注册写在组件的 `directives` 配置中，只能在当前组件使用：

```vue
<script>
export default {
  directives: {
    focus: {
      inserted (el) {
        el.focus()
      }
    }
  }
}
</script>
```

使用时必须先注册再使用：

```html
<input v-focus type="text">
```

### 2.3 `inserted` 与 `el`

Vue 2 的 `inserted` 钩子会在绑定元素插入父节点后触发。回调的第一个参数 `el` 就是使用指令的真实 DOM 元素，可以调用 `focus()`、设置样式、添加类名等。

### 2.4 带值的自定义指令

指令可以通过等号接收值：

```html
<h2 v-color="color1">第一种颜色</h2>
<h2 v-color="color2">第二种颜色</h2>
```

在指令钩子中通过 `binding.value` 获取值：

```js
directives: {
  color: {
    inserted (el, binding) {
      el.style.color = binding.value
    },
    update (el, binding) {
      el.style.color = binding.value
    }
  }
}
```

如果指令值会变化，需要在 `update` 中同步更新 DOM。也可以在 `bind`、`componentUpdated` 等钩子中补充更细的处理。

### 2.5 封装 `v-loading`

请求还没有返回时，如果页面完全空白，用户很难判断程序是否正在工作。`v-loading` 可以给容器加一个加载蒙层。

实现思路：

1. 准备一个 `.loading` 类，用伪元素覆盖容器。
2. 指令值为 `true` 时添加类，为 `false` 时移除类。
3. 在请求开始和结束时切换响应式的 `isLoading`。

```vue
<template>
  <div v-loading="isLoading" class="box">
    <ul>
      <li v-for="item in list" :key="item.id">{{ item.title }}</li>
    </ul>
  </div>
</template>

<script>
import axios from 'axios'

export default {
  directives: {
    loading: {
      inserted (el, binding) {
        el.classList.toggle('loading', binding.value)
      },
      update (el, binding) {
        el.classList.toggle('loading', binding.value)
      }
    }
  },
  data () {
    return { list: [], isLoading: false }
  },
  async created () {
    this.isLoading = true
    try {
      const res = await axios.get('http://hmajax.itheima.net/api/news')
      this.list = res.data.data
    } finally {
      this.isLoading = false
    }
  }
}
</script>

<style>
.box { position: relative; min-height: 300px; border: 2px solid orange; }
.loading::before {
  content: '';
  position: absolute;
  inset: 0;
  z-index: 1;
  background: rgba(255, 255, 255, .9) url('./loading.gif') no-repeat center;
}
</style>
```

指令只负责“如何显示加载状态”，请求状态仍然由组件的 `data` 管理，这样职责更清楚。

## 三、默认插槽与后备内容

### 3.1 默认插槽

如果组件内部某块结构不确定，可以用 `<slot></slot>` 占位，使用组件时在标签内部传入内容：

```vue
<!-- MyDialog.vue -->
<template>
  <div class="dialog">
    <h3>友情提示</h3>
    <div class="dialog-content">
      <slot></slot>
    </div>
  </div>
</template>
```

```vue
<!-- 使用方 -->
<MyDialog>
  <p>你确定要删除这条记录吗？</p>
</MyDialog>
```

插槽内容可以是纯文本、HTML 标签或其他组件。组件负责外框结构，使用方负责填入具体内容。

### 3.2 后备内容

如果调用方没有传入内容，`slot` 标签内部的内容会作为默认值：

```vue
<div class="dialog-content">
  <slot>这里是默认提示内容</slot>
</div>
```

```vue
<MyDialog />
<!-- 显示默认提示内容 -->

<MyDialog>你确认要退出吗？</MyDialog>
<!-- 显示外部传入的内容 -->
```

## 四、具名插槽

### 4.1 解决多个可定制区域

默认插槽只能对应一个位置。当组件中有头部、内容、底部多个区域时，给每个插槽设置 `name`：

```vue
<!-- MyDialog.vue -->
<template>
  <div class="dialog">
    <header><slot name="header">默认标题</slot></header>
    <main><slot name="content">默认内容</slot></main>
    <footer><slot name="footer">默认按钮</slot></footer>
  </div>
</template>
```

使用方通过 `v-slot:插槽名` 分发内容：

```vue
<MyDialog>
  <template v-slot:header>
    <h3>删除商品</h3>
  </template>
  <template v-slot:content>
    <p>删除后无法恢复，是否继续？</p>
  </template>
  <template #footer>
    <button>取消</button>
    <button>确认</button>
  </template>
</MyDialog>
```

`v-slot:footer` 可以简写为 `#footer`。未命名的 `<slot>` 是默认插槽，插槽名为 `default`。

## 五、作用域插槽

### 5.1 作用

普通插槽由父组件传入结构，但如果父组件还需要使用子组件内部循环产生的数据，就需要作用域插槽。

子组件在 `slot` 上绑定属性：

```vue
<slot name="body" :item="item" :index="index"></slot>
```

这些属性会收集为一个对象，父组件通过 `#body="scope"` 接收：

```vue
<template #body="scope">
  <td>{{ scope.index + 1 }}</td>
  <td>{{ scope.item.name }}</td>
</template>
```

也可以直接解构：`<template #body="{ item, index }">`。

### 5.2 可复用表格组件

```vue
<!-- MyTable.vue -->
<template>
  <table class="my-table">
    <thead><tr><slot name="head"></slot></tr></thead>
    <tbody>
      <tr v-for="(item, index) in data" :key="item.id">
        <slot name="body" :item="item" :index="index"></slot>
      </tr>
    </tbody>
  </table>
</template>

<script>
export default {
  props: {
    data: { type: Array, required: true }
  }
}
</script>
```

```vue
<!-- App.vue -->
<MyTable :data="goods">
  <template #head>
    <th>编号</th><th>名称</th><th>图片</th><th>标签</th>
  </template>
  <template #body="{ item, index }">
    <td>{{ index + 1 }}</td>
    <td>{{ item.name }}</td>
    <td><img :src="item.picture" alt=""></td>
    <td><MyTag v-model="item.tag" /></td>
  </template>
</MyTable>
```

`MyTable` 负责遍历和表格骨架，外部决定表头和每一行的具体内容，这就是作用域插槽的核心价值。

## 六、综合案例：商品列表组件封装

### 6.1 `MyTag`：可编辑标签

需求：双击标签显示输入框，输入框自动聚焦；失去焦点退出编辑；按回车提交修改；通过 `v-model` 和父组件同步。

```vue
<template>
  <div class="my-tag">
    <input
      v-if="isEdit"
      v-focus
      ref="input"
      :value="value"
      @blur="isEdit = false"
      @keyup.enter="submit"
    >
    <div v-else class="text" @dblclick="startEdit">{{ value }}</div>
  </div>
</template>

<script>
export default {
  props: { value: { type: String, default: '' } },
  data () { return { isEdit: false } },
  methods: {
    startEdit () { this.isEdit = true },
    submit (event) {
      const value = event.target.value.trim()
      if (!value) return alert('标签内容不能为空')
      this.$emit('input', value)
      this.isEdit = false
    }
  }
}
</script>
```

这里把 `v-focus`、`v-model`、`ref` 和事件结合起来：指令负责聚焦，`value/input` 负责组件双向绑定，组件自己的 `isEdit` 负责显示状态。

### 6.2 `MyTable`：动态数据和自定义结构

```vue
<!-- App.vue -->
<MyTable :data="goods">
  <template #head>
    <th>编号</th><th>名称</th><th>图片</th><th>标签</th>
  </template>
  <template #body="{ item, index }">
    <td>{{ index + 1 }}</td>
    <td>{{ item.name }}</td>
    <td><img :src="item.picture" alt="商品图"></td>
    <td><MyTag v-model="item.tag" /></td>
  </template>
</MyTable>
```

```js
// 商品数据由父组件维护
data () {
  return {
    goods: [
      { id: 101, name: '紫砂壶', picture: '...', tag: '茶具' },
      { id: 102, name: '户外徒步鞋', picture: '...', tag: '男鞋' }
    ]
  }
}
```

当 `MyTag` 触发 `input` 事件时，`v-model="item.tag"` 会把新标签写回父组件的商品数据。父组件仍然是数据的所有者。

## 七、单页应用程序 SPA

### 7.1 概念

SPA（Single Page Application）把主要功能放在一个 HTML 页面中，切换功能时只更新页面中的局部组件，不重新加载整个 HTML。

- 单页应用常见于管理系统、内部系统、文档站和移动端站点。
- 多页应用常见于公司官网、电商网站等需要独立页面和 SEO 的场景。

### 7.2 SPA 的优势和代价

优势：

- 页面切换速度快，公共区域不重复加载。
- 组件和状态可以复用。
- 前后端职责更清晰。

代价：

- 首次加载需要下载 JS 资源。
- 需要处理路由、刷新、权限和错误页面。
- SEO 和首屏性能需要额外优化。

SPA 想做到“地址变了，局部组件也切换”，需要建立访问路径和组件之间的映射关系，这就是路由。

## 八、Vue Router 基本使用

### 8.1 Vue 中的路由

Vue Router 是 Vue 官方路由插件，负责把 URL 路径映射到页面组件：

```text
/#/find   → Find.vue
/#/my     → My.vue
/#/friend → Friend.vue
```

课程使用 Vue 2 对应的 Vue Router 3.6.5：

```bash
yarn add vue-router@3.6.5
# 或 npm install vue-router@3.6.5
```

### 8.2 使用步骤：5 + 2

#### 5 个基础步骤

1. 安装 `vue-router`。
2. 在 `main.js` 中引入 `VueRouter`。
3. `Vue.use(VueRouter)` 安装插件。
4. 创建 `new VueRouter()` 路由对象。
5. 将路由对象注入根 Vue 实例。

```js
// main.js
import Vue from 'vue'
import App from './App.vue'
import VueRouter from 'vue-router'

Vue.use(VueRouter)
const router = new VueRouter()

new Vue({
  render: h => h(App),
  router
}).$mount('#app')
```

#### 2 个核心步骤

6. 创建页面组件并配置路由规则。
7. 在模板中配置导航和路由出口。

### 8.3 路由规则、导航和出口

```js
// router/index.js
import Vue from 'vue'
import VueRouter from 'vue-router'
import Find from '@/views/Find.vue'
import My from '@/views/My.vue'
import Friend from '@/views/Friend.vue'

Vue.use(VueRouter)

export default new VueRouter({
  routes: [
    { path: '/', redirect: '/find' },
    { path: '/find', component: Find },
    { path: '/my', component: My },
    { path: '/friend', component: Friend }
  ]
})
```

```vue
<!-- App.vue -->
<template>
  <div id="app">
    <nav>
      <router-link to="/find">发现音乐</router-link>
      <router-link to="/my">我的音乐</router-link>
      <router-link to="/friend">朋友</router-link>
    </nav>
    <main>
      <router-view></router-view>
    </main>
  </div>
</template>
```

- `router-link` 会生成导航链接并处理路由跳转。
- `router-view` 是路由组件的显示出口。
- `redirect` 可以把根路径重定向到默认页面。

## 九、页面组件与复用组件

`.vue` 文件本质上没有区别，但按用途分目录可以让项目更易维护：

```text
src/
├─ views/       页面组件，通常与路由规则对应
│  ├─ Find.vue
│  ├─ My.vue
│  └─ Friend.vue
├─ components/  复用组件，多个页面都可能使用
│  ├─ MyTag.vue
│  └─ MyTable.vue
└─ router/
   └─ index.js
```

- `views` 放“页面级”组件，负责组合页面模块。
- `components` 放“复用级”组件，负责提供可复用的 UI 或功能。
- `@` 在脚手架中通常指向 `src`，可以写 `@/views/Find.vue`，减少相对路径层级。

## 十、路由模块抽离

把所有路由写在 `main.js` 会让入口文件越来越长。推荐把路由配置放到 `src/router/index.js`：

```js
// src/router/index.js
import Vue from 'vue'
import VueRouter from 'vue-router'
import Home from '@/views/Home.vue'
import Search from '@/views/Search.vue'

Vue.use(VueRouter)

const router = new VueRouter({
  mode: 'hash',
  routes: [
    { path: '/', redirect: '/home' },
    { path: '/home', component: Home },
    { path: '/search', component: Search }
  ]
})

export default router
```

```js
// src/main.js
import Vue from 'vue'
import App from './App.vue'
import router from './router'

new Vue({
  router,
  render: h => h(App)
}).$mount('#app')
```

模块抽离后，入口只负责启动应用，路由文件负责路径和组件的关系，后续添加权限、懒加载、404 页面时更容易维护。

## 十一、常见错误与排查

### 11.1 自定义指令使用时报错

检查指令是否先注册，注册名是否省略了 `v-`，使用时是否写了 `v-`。

### 11.2 插槽没有显示

检查 `slot` 的 `name` 是否一致，具名插槽是否使用了 `v-slot:name` 或 `#name`。默认插槽不要误写成具名插槽。

### 11.3 作用域插槽拿不到数据

子组件必须在 `<slot>` 上绑定数据，父组件要接收作用域对象：

```vue
<slot name="body" :item="item"></slot>
<template #body="{ item }">{{ item.name }}</template>
```

### 11.4 路由页面不切换

依次检查：是否安装插件、是否 `Vue.use(VueRouter)`、是否把 `router` 注入根实例、是否有 `<router-view>`、路径是否匹配规则。

### 11.5 路由链接刷新页面

优先使用 `<router-link>` 或 `this.$router.push()`，不要用普通 `<a href>` 触发整页刷新。

### 11.6 组件双向绑定不生效

Vue 2 默认检查 `value` prop 和 `input` 事件；如果使用 `.sync`，检查是否触发了 `update:属性名`。

## 十二、练习题

1. 写一个 `v-focus` 全局指令，让登录页面的用户名输入框自动聚焦。
2. 写一个 `v-color` 指令，监听指令值变化并更新文字颜色。
3. 封装 `MyDialog`，用默认插槽、具名插槽和后备内容完成标题、内容、底部按钮的定制。
4. 封装 `MyTable`，用作用域插槽允许调用方自定义表头和表格行。
5. 完成 `MyTag` 的双击编辑、回车提交、空值校验和 `v-model` 双向绑定。
6. 新建 `Find`、`My`、`Friend` 三个页面组件，完成 Vue Router 的 5 + 2 基本配置。
7. 把路由配置从 `main.js` 抽离到 `router/index.js`，为根路径增加重定向。

## 十三、当天速记

- 自定义指令封装元素级 DOM 操作，注册时不写 `v-`，使用时要写 `v-`。
- 指令值通过 `binding.value` 获取，值变化时在 `update` 中同步。
- 默认插槽只有一个位置；具名插槽用 `name` 和 `#name`；作用域插槽用 slot 属性把数据传给外部结构。
- 商品列表案例把 `MyTag`、`MyTable`、自定义指令和作用域插槽组合起来。
- SPA 把功能放在一个 HTML 页面中，通过路由按需切换组件。
- Vue Router 的核心是“路径和组件的映射”，`router-link` 负责导航，`router-view` 负责显示。
- 页面组件放 `views`，复用组件放 `components`，路由配置抽到 `router/index.js`。